In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import tensorflow as tf
from skimage.metrics import structural_similarity as ssim

In [2]:
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [3]:
def find_matching_files(pred_dir, obs_dir):
    file_pairs = []
    
    #Extract and sort prediction files
    pred_files = []
    for f in os.listdir(pred_dir):
        match = re.match(r'^(\d+)\.(png|jpg|jpeg)$', f, re.IGNORECASE)
        if match:
            pred_files.append((int(match.group(1)), f))
    pred_files.sort(key=lambda x: x[0])
    
    #Extract and sort observation files
    obs_files = []
    for f in os.listdir(obs_dir):
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            match = re.search(r'(\d+)', f)
            if match:
                obs_files.append((int(match.group(1)), f))
            else:
                obs_files.append((len(obs_files) + 1, f))
    obs_files.sort(key=lambda x: x[0])
    
    #Pair files by order
    min_length = min(len(pred_files), len(obs_files))
    for i in range(min_length):
        file_pairs.append({
            'timestep': i + 1,
            'pred_path': os.path.join(pred_dir, pred_files[i][1]),
            'obs_path': os.path.join(obs_dir, obs_files[i][1])
        })
    
    if len(pred_files) != len(obs_files):
        print(f"Warning: File count mismatch. Using first {min_length} pairs.")
    
    return file_pairs

In [4]:
def load_and_preprocess(image_path, target_size=None, for_ms_ssim=False):
    try:
        img = cv2.imread(image_path)
        if img is None:
            raise FileNotFoundError(f"Image not found: {image_path}")
        
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        if target_size:
            img_rgb = cv2.resize(img_rgb, target_size, interpolation=cv2.INTER_AREA)
        
        if for_ms_ssim:
            return img_rgb
        else:
            gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
            return img_rgb, gray
    except Exception as e:
        print(f"Error processing {image_path}: {str(e)}")
        return None if for_ms_ssim else (None, None)

In [5]:
def calculate_ms_ssim(img1, img2):
    if img1 is None or img2 is None:
        return 0.0
    
    try:
        #Convert to numpy arrays if they're not already
        img1 = np.array(img1) if not isinstance(img1, np.ndarray) else img1
        img2 = np.array(img2) if not isinstance(img2, np.ndarray) else img2
        
        #Ensure identical dimensions
        min_height = min(img1.shape[0], img2.shape[0])
        min_width = min(img1.shape[1], img2.shape[1])
        img1 = img1[:min_height, :min_width, :]
        img2 = img2[:min_height, :min_width, :]
        
        #Normalize to [0, 1] range
        img1_normalized = img1 / 255.0
        img2_normalized = img2 / 255.0
        
        #Calculate MS-SSIM
        return float(tf.image.ssim_multiscale(
            tf.convert_to_tensor(img1_normalized, dtype=tf.float32),
            tf.convert_to_tensor(img2_normalized, dtype=tf.float32),
            max_val=1.0
        ).numpy())
    except Exception as e:
        print(f"MS-SSIM calculation failed: {str(e)}")
        return 0.0


In [6]:
def calculate_ssim(img1, img2):
    if img1 is None or img2 is None:
        return 0.0
    
    try:
        #Convert to numpy arrays if they're not already
        img1 = np.array(img1) if not isinstance(img1, np.ndarray) else img1
        img2 = np.array(img2) if not isinstance(img2, np.ndarray) else img2
        
        if img1.shape != img2.shape:
            min_height = min(img1.shape[0], img2.shape[0])
            min_width = min(img1.shape[1], img2.shape[1])
            img1 = img1[:min_height, :min_width]
            img2 = img2[:min_height, :min_width]
        
        return ssim(img1, img2, data_range=img2.max() - img2.min())
    except Exception as e:
        print(f"❌ SSIM calculation failed: {str(e)}")
        return 0.0


In [7]:
def create_comparison_figure(pred_img, obs_img, pred_title, obs_title, ssim_val, timestep):
    fig = plt.figure(figsize=(14, 7))
    
    #Prediction subplot
    ax1 = plt.subplot(1, 2, 1)
    ax1.imshow(pred_img)
    ax1.set_title(pred_title, fontsize=12, pad=10)
    ax1.axis('off')
    for spine in ax1.spines.values():
        spine.set_edgecolor('#3498db')
        spine.set_linewidth(3)
    
    #Observation subplot
    ax2 = plt.subplot(1, 2, 2)
    ax2.imshow(obs_img)
    ax2.set_title(obs_title, fontsize=12, pad=10)
    ax2.axis('off')
    for spine in ax2.spines.values():
        spine.set_edgecolor('#e74c3c')
        spine.set_linewidth(3)
    
    #Annotations
    fig.text(0.5, 0.92, f"MS-SSIM: {ssim_val:.3f}", 
             ha='center', va='center', fontsize=14,
             bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray'))
    fig.text(0.5, 0.05, f"Timestep: +{timestep} minutes", 
             ha='center', va='center', fontsize=12)
    
    plt.tight_layout(rect=[0, 0.05, 1, 0.93])
    return fig


In [8]:
def create_timeseries_plot(timesteps, ssim_values):
    fig, ax = plt.subplots(figsize=(12, 6), dpi=100)
    
    #Main plot line with enhanced styling
    main_line = ax.plot(
        timesteps, 
        ssim_values, 
        'o-',
        color='#2c7bb6',
        markersize=9,
        linewidth=2.5,
        markerfacecolor='white',
        markeredgewidth=2,
        markeredgecolor='#2c7bb6',
        zorder=3
    )
    
    #Reference zones with improved colors
    ref_zones = [
        (0.8, 1.0, '#d9f0a3', 'Excellent'),
        (0.5, 0.8, '#fee08b', 'Good'), 
        (0.0, 0.5, '#fc8d59', 'Poor')
    ]
    
    for ymin, ymax, color, label in ref_zones:
        ax.axhspan(
            ymin, ymax, 
            facecolor=color, 
            alpha=0.2,
            zorder=1
        )
    
    #Custom grid lines
    ax.grid(
        True, 
        which='both', 
        linestyle='--', 
        linewidth=0.7,
        alpha=0.4,
        zorder=2
    )
    
    #Enhanced axis formatting
    ax.set(
        xlabel='Timestep (minutes)',
        ylabel='SSIM Value',
        title='Structural Similarity Over Time',
        ylim=(0, 1.05),
        xlim=(min(timesteps)-0.5, max(timesteps)+0.5)
    )
    
    #Perfect tick placement
    ax.set_xticks(timesteps)
    ax.set_xticklabels([t*2 for t in timesteps])
    ax.set_yticks(np.arange(0, 1.1, 0.1))
    
    #Value annotations with perfect positioning
    for i, val in enumerate(ssim_values):
        ax.annotate(
            f"{val:.2f}",
            (timesteps[i], val),
            textcoords="offset points",
            xytext=(0, 10),
            ha='center',
            fontsize=10,
            bbox=dict(
                boxstyle='round,pad=0.3',
                fc='white',
                ec='none',
                alpha=0.8
            ),
            zorder=4
        )
    
    # Perfect legend
    ax.legend(
        ['SSIM Score'] + [label for _, _, _, label in ref_zones],
        loc='upper center',
        bbox_to_anchor=(0.5, -0.15),
        ncol=4,
        frameon=True,
        framealpha=0.9,
        fontsize=10
    )
    
    # Optimal layout
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.2)
    
    return fig


In [9]:
def process_time_series(pred_dir, obs_dir, output_dir, use_ms_ssim=True):
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'comparisons'), exist_ok=True)
    
    file_pairs = find_matching_files(pred_dir, obs_dir)
    if not file_pairs:
        print("No matching files found! Check directories:")
        print(f"Predictions: {os.listdir(pred_dir)}")
        print(f"Observations: {os.listdir(obs_dir)}")
        return [], []
    
    timesteps, ssim_values = [], []
    metric_name = "MS-SSIM" if use_ms_ssim else "SSIM"
    
    for pair in file_pairs:
        # Load images
        if use_ms_ssim:
            pred_img = load_and_preprocess(pair['pred_path'], for_ms_ssim=True)
            obs_img = load_and_preprocess(pair['obs_path'], for_ms_ssim=True)
            similarity_val = calculate_ms_ssim(pred_img, obs_img)
        else:
            pred_img, pred_gray = load_and_preprocess(pair['pred_path'])
            obs_img, obs_gray = load_and_preprocess(pair['obs_path'])
            similarity_val = calculate_ssim(pred_gray, obs_gray)
        
        print(f"Timestep {pair['timestep']}: {metric_name} = {similarity_val:.4f}")
        
        # Save comparison figure
        fig = create_comparison_figure(
            pred_img if use_ms_ssim else pred_img,
            obs_img if use_ms_ssim else obs_img,
            f"Nowcast (Timestep {pair['timestep']})",
            "Observation",
            similarity_val,
            pair['timestep']
        )
        fig.savefig(
            os.path.join(output_dir, 'comparisons', f'comparison_{pair["timestep"]:02d}.png'),
            bbox_inches='tight', dpi=150
        )
        plt.close(fig)
        
        timesteps.append(pair['timestep'])
        ssim_values.append(similarity_val)
    
    # Save time series plot
    if timesteps:
        ts_fig = create_timeseries_plot(timesteps, ssim_values)
        ts_fig.savefig(
            os.path.join(output_dir, f'{metric_name.lower()}_timeseries.png'),
            bbox_inches='tight', dpi=150
        )
        plt.close(ts_fig)
        
        # Export CSV results
        with open(os.path.join(output_dir, f'{metric_name.lower()}_results.csv'), 'w') as f:
            f.write(f"Timestep,{metric_name}\n")
            f.writelines([f"{t},{s:.4f}\n" for t, s in zip(timesteps, ssim_values)])
    
    return timesteps, ssim_values

In [10]:
if __name__ == "__main__":
    # Configure paths
    BASE_DIR = "D:/College/MENEA/FIXED_DATA/OUTPUT/20200813/"
    PRED_DIR = os.path.join(BASE_DIR, 'nowcasts')
    OBS_DIR = os.path.join(BASE_DIR, 'observations')
    OUTPUT_DIR = os.path.join(BASE_DIR, 'shapematch')
    
    print("\n" + "="*50)
    print("Nowcast-Observation Comparison Tool")
    print("="*50)
    
    try:
        timesteps, scores = process_time_series(
            pred_dir=PRED_DIR,
            obs_dir=OBS_DIR,
            output_dir=OUTPUT_DIR,
            use_ms_ssim=True  # Set False to use standard SSIM
        )
        
        if timesteps:
            plt.show()  # Display final plot
            print(f"Results saved to: {OUTPUT_DIR}")
        else:
            print("No valid comparisons generated")
            
    except Exception as e:
        print(f"Critical error: {str(e)}")


Nowcast-Observation Comparison Tool
Timestep 1: MS-SSIM = 0.4495
Timestep 2: MS-SSIM = 0.4501
Timestep 3: MS-SSIM = 0.4578
Timestep 4: MS-SSIM = 0.4691
Timestep 5: MS-SSIM = 0.4771
Timestep 6: MS-SSIM = 0.4833
Timestep 7: MS-SSIM = 0.4874
Timestep 8: MS-SSIM = 0.4848
Timestep 9: MS-SSIM = 0.4854
Timestep 10: MS-SSIM = 0.4839
Timestep 11: MS-SSIM = 0.4869
Timestep 12: MS-SSIM = 0.4863
Timestep 13: MS-SSIM = 0.4873
Timestep 14: MS-SSIM = 0.4865
Timestep 15: MS-SSIM = 0.4956
Timestep 16: MS-SSIM = 0.5000
Timestep 17: MS-SSIM = 0.4943
Timestep 18: MS-SSIM = 0.4934
Timestep 19: MS-SSIM = 0.4883
Timestep 20: MS-SSIM = 0.4981
Timestep 21: MS-SSIM = 0.4953
Timestep 22: MS-SSIM = 0.4819
Timestep 23: MS-SSIM = 0.4772
Timestep 24: MS-SSIM = 0.4800
Timestep 25: MS-SSIM = 0.4818
Timestep 26: MS-SSIM = 0.4743
Timestep 27: MS-SSIM = 0.4657
Timestep 28: MS-SSIM = 0.4629
Timestep 29: MS-SSIM = 0.4598
Timestep 30: MS-SSIM = 0.4534
Results saved to: D:/College/MENEA/FIXED_DATA/OUTPUT/20200813/shapematch

In [5]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import tensorflow as tf
from skimage.metrics import structural_similarity as ssim
from skimage.measure import label, regionprops

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

def find_matching_files(pred_dir, obs_dir):
    """Find matching prediction and observation files with timestamp pairing"""
    file_pairs = []
    
    # Extract and sort prediction files by sequence number
    pred_files = []
    for f in os.listdir(pred_dir):
        match = re.match(r'^(\d+)\.(png|jpg|jpeg)$', f, re.IGNORECASE)
        if match:
            sequence_num = int(match.group(1))
            pred_files.append((sequence_num, f))
    pred_files.sort(key=lambda x: x[0])
    
    if not pred_files:
        print(f"No prediction files found in {pred_dir}")
        return file_pairs
    
    # Extract and sort observation files by timestamp
    obs_files = []
    for f in os.listdir(obs_dir):
        match = re.search(r'_(\d{6})\.(png|jpg|jpeg)$', f, re.IGNORECASE)
        if match:
            # Convert HHMMSS timestamp to seconds since first observation
            time_str = match.group(1)
            timestamp = (int(time_str[:2]) * 3600 + 
                         int(time_str[2:4]) * 60 + 
                         int(time_str[4:6]))
            obs_files.append((timestamp, f))
    obs_files.sort(key=lambda x: x[0])
    
    if not obs_files:
        print(f"No observation files found in {obs_dir}")
        return file_pairs
    
    # Calculate time between observations (assuming regular intervals)
    if len(obs_files) > 1:
        time_step = obs_files[1][0] - obs_files[0][0]
    else:
        time_step = 120  # default 2 minutes if only one observation
    
    # Pair prediction sequence numbers with observation times
    first_obs_time = obs_files[0][0]
    
    for seq_num, pred_file in pred_files:
        # Calculate predicted time (assuming predictions are at same intervals)
        pred_time = first_obs_time + (seq_num - 1) * time_step
        
        # Find closest observation
        closest_obs = min(obs_files, key=lambda x: abs(x[0] - pred_time))
        time_diff = abs(closest_obs[0] - pred_time)
        
        file_pairs.append({
            'timestep': seq_num,  # Using sequence number as timestep
            'pred_path': os.path.join(pred_dir, pred_file),
            'obs_path': os.path.join(obs_dir, closest_obs[1]),
            'time_diff': time_diff,
            'pred_time': pred_time,
            'obs_time': closest_obs[0],
            'obs_filename': closest_obs[1]
        })
    
    # Filter bad matches (more than half the interval)
    max_time_diff = time_step / 2
    file_pairs = [p for p in file_pairs if p['time_diff'] <= max_time_diff]
    
    if len(file_pairs) < len(pred_files):
        print(f"Warning: Could only match {len(file_pairs)}/{len(pred_files)} prediction files")
    
    # Debug output
    print("\nMatched files:")
    for pair in file_pairs:
        print(f"Pred {pair['timestep']} -> Obs {pair['obs_filename']} (Δ {pair['time_diff']}s)")
    
    return file_pairs

def load_and_preprocess(image_path, target_size=None):
    """Load and preprocess image with validation"""
    try:
        img = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
        if img is None:
            raise ValueError(f"Failed to load image: {image_path}")
        
        # Handle different channel counts
        if len(img.shape) == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        elif img.shape[2] == 4:
            img = cv2.cvtColor(img, cv2.COLOR_BGRA2RGB)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        if target_size:
            img = cv2.resize(img, target_size, interpolation=cv2.INTER_AREA)
        
        return img
    except Exception as e:
        print(f"Error loading {image_path}: {str(e)}")
        return None

def calculate_ssim(img1, img2):
    """Calculate SSIM with validation and dynamic data range"""
    if img1 is None or img2 is None:
        return 0.0
    
    try:
        # Convert to grayscale if needed
        if len(img1.shape) == 3:
            img1 = cv2.cvtColor(img1, cv2.COLOR_RGB2GRAY)
        if len(img2.shape) == 3:
            img2 = cv2.cvtColor(img2, cv2.COLOR_RGB2GRAY)
        
        # Ensure identical dimensions
        min_height = min(img1.shape[0], img2.shape[0])
        min_width = min(img1.shape[1], img2.shape[1])
        img1 = img1[:min_height, :min_width]
        img2 = img2[:min_height, :min_width]
        
        # Calculate dynamic data range
        data_range = max(img2.max() - img2.min(), img1.max() - img1.min())
        if data_range == 0:
            return 0.0
        
        return ssim(img1, img2, data_range=data_range)
    except Exception as e:
        print(f"SSIM calculation failed: {str(e)}")
        return 0.0

def calculate_ms_ssim(img1, img2):
    """Calculate MS-SSIM with proper validation"""
    if img1 is None or img2 is None:
        return 0.0
    
    try:
        # Ensure 3-channel color images
        if len(img1.shape) == 2:
            img1 = cv2.cvtColor(img1, cv2.COLOR_GRAY2RGB)
        if len(img2.shape) == 2:
            img2 = cv2.cvtColor(img2, cv2.COLOR_GRAY2RGB)
        
        # Ensure identical dimensions
        min_height = min(img1.shape[0], img2.shape[0])
        min_width = min(img1.shape[1], img2.shape[1])
        img1 = img1[:min_height, :min_width, :]
        img2 = img2[:min_height, :min_width, :]
        
        # Normalize to [0, 1] range
        img1 = img1.astype(np.float32) / 255.0
        img2 = img2.astype(np.float32) / 255.0
        
        # Calculate MS-SSIM
        return tf.image.ssim_multiscale(
            tf.convert_to_tensor(img1),
            tf.convert_to_tensor(img2),
            max_val=1.0
        ).numpy()
    except Exception as e:
        print(f"MS-SSIM calculation failed: {str(e)}")
        return 0.0

def create_comparison_figure(pred_img, obs_img, pred_title, obs_title, ssim_val, timestep):
    """Create side-by-side comparison figure with original style"""
    fig = plt.figure(figsize=(14, 7))
    
    # Prediction subplot
    ax1 = plt.subplot(1, 2, 1)
    ax1.imshow(pred_img)
    ax1.set_title(pred_title, fontsize=12, pad=10)
    ax1.axis('off')
    for spine in ax1.spines.values():
        spine.set_edgecolor('#3498db')
        spine.set_linewidth(3)
    
    # Observation subplot
    ax2 = plt.subplot(1, 2, 2)
    ax2.imshow(obs_img)
    ax2.set_title(obs_title, fontsize=12, pad=10)
    ax2.axis('off')
    for spine in ax2.spines.values():
        spine.set_edgecolor('#e74c3c')
        spine.set_linewidth(3)
    
    # Annotations
    fig.text(0.5, 0.92, f"MS-SSIM: {ssim_val:.3f}", 
             ha='center', va='center', fontsize=14,
             bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray'))
    fig.text(0.5, 0.05, f"Timestep: {timestep}", 
             ha='center', va='center', fontsize=12)
    
    plt.tight_layout(rect=[0, 0.05, 1, 0.93])
    return fig

def create_timeseries_plot(timesteps, ssim_values):
    """Create time series plot with original style"""
    fig, ax = plt.subplots(figsize=(12, 6), dpi=100)
    
    # Main plot line with enhanced styling
    main_line = ax.plot(
        timesteps, 
        ssim_values, 
        'o-',
        color='#2c7bb6',
        markersize=9,
        linewidth=2.5,
        markerfacecolor='white',
        markeredgewidth=2,
        markeredgecolor='#2c7bb6',
        zorder=3
    )
    
    # Reference zones with improved colors
    ref_zones = [
        (0.8, 1.0, '#d9f0a3', 'Excellent'),
        (0.5, 0.8, '#fee08b', 'Good'), 
        (0.0, 0.5, '#fc8d59', 'Poor')
    ]
    
    for ymin, ymax, color, label in ref_zones:
        ax.axhspan(
            ymin, ymax, 
            facecolor=color, 
            alpha=0.2,
            zorder=1
        )
    
    # Custom grid lines
    ax.grid(
        True, 
        which='both', 
        linestyle='--', 
        linewidth=0.7,
        alpha=0.4,
        zorder=2
    )
    
    # Enhanced axis formatting
    ax.set(
        xlabel='Timestep',
        ylabel='SSIM Value',
        title='Structural Similarity Over Time',
        ylim=(0, 1.05),
        xlim=(min(timesteps)-0.5, max(timesteps)+0.5)
    )
    
    # Value annotations with perfect positioning
    for i, val in enumerate(ssim_values):
        ax.annotate(
            f"{val:.2f}",
            (timesteps[i], val),
            textcoords="offset points",
            xytext=(0, 10),
            ha='center',
            fontsize=10,
            bbox=dict(
                boxstyle='round,pad=0.3',
                fc='white',
                ec='none',
                alpha=0.8
            ),
            zorder=4
        )
    
    plt.tight_layout()
    return fig

def process_time_series(pred_dir, obs_dir, output_dir):
    """Process all image pairs with improved matching"""
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'comparisons'), exist_ok=True)
    
    file_pairs = find_matching_files(pred_dir, obs_dir)
    if not file_pairs:
        print("No matching files found! Check directories:")
        print(f"Predictions: {os.listdir(pred_dir)}")
        print(f"Observations: {os.listdir(obs_dir)}")
        return [], []
    
    timesteps, ssim_values = [], []
    
    for pair in file_pairs:
        # Load images
        pred_img = load_and_preprocess(pair['pred_path'])
        obs_img = load_and_preprocess(pair['obs_path'])
        
        if pred_img is None or obs_img is None:
            print(f"Skipping pair {pair['timestep']} due to loading error")
            continue
        
        # Calculate both SSIM and MS-SSIM
        ssim_val = calculate_ssim(pred_img, obs_img)
        ms_ssim_val = calculate_ms_ssim(pred_img, obs_img)
        
        print(f"Timestep {pair['timestep']}: SSIM={ssim_val:.4f}, MS-SSIM={ms_ssim_val:.4f}")
        
        # Save comparison figure using MS-SSIM
        fig = create_comparison_figure(
            pred_img,
            obs_img,
            f"Nowcast (Sequence {pair['timestep']})",
            f"Observation (Time: {pair['obs_time']}, Δ {pair['time_diff']}s)",
            ms_ssim_val,
            pair['timestep']
        )
        fig.savefig(
            os.path.join(output_dir, 'comparisons', f'comparison_{pair["timestep"]}.png'),
            bbox_inches='tight', dpi=150
        )
        plt.close(fig)
        
        timesteps.append(pair['timestep'])
        ssim_values.append(ms_ssim_val)
    
    # Save time series plot
    if timesteps:
        ts_fig = create_timeseries_plot(timesteps, ssim_values)
        ts_fig.savefig(
            os.path.join(output_dir, 'ms_ssim_timeseries.png'),
            bbox_inches='tight', dpi=150
        )
        plt.close(ts_fig)
        
        # Export CSV results
        with open(os.path.join(output_dir, 'results.csv'), 'w') as f:
            f.write("Timestep,SSIM,MS-SSIM,ObservationTime,TimeDifference\n")
            for p in file_pairs:
                f.write(f"{p['timestep']},{ssim(p['timestep']):.4f},{ssim_values[timesteps.index(p['timestep'])]:.4f},{p['obs_time']},{p['time_diff']}\n")
    
    return timesteps, ssim_values

if __name__ == "__main__":
    # Configure paths
    BASE_DIR = "D:/College/MENEA/FIXED_DATA/OUTPUT/20200228/"
    PRED_DIR = os.path.join(BASE_DIR, 'nowcasts')
    OBS_DIR = os.path.join(BASE_DIR, 'observations')
    OUTPUT_DIR = os.path.join(BASE_DIR, 'shapematch')
    
    print("\n" + "="*50)
    print("Enhanced Nowcast Verification Tool")
    print("="*50)
    
    # Verify directories exist
    print(f"\nPrediction directory: {PRED_DIR}")
    print(f"Observation directory: {OBS_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
    
    try:
        timesteps, scores = process_time_series(
            pred_dir=PRED_DIR,
            obs_dir=OBS_DIR,
            output_dir=OUTPUT_DIR
        )
        
        if timesteps:
            plt.show()  # Display final plot
            print(f"\nResults saved to: {OUTPUT_DIR}")
            print(f"Processed {len(timesteps)} time steps successfully")
        else:
            print("\nNo valid comparisons generated")
            
    except Exception as e:
        print(f"\nCritical error: {str(e)}")


Enhanced Nowcast Verification Tool

Prediction directory: D:/College/MENEA/FIXED_DATA/OUTPUT/20200228/nowcasts
Observation directory: D:/College/MENEA/FIXED_DATA/OUTPUT/20200228/observations
Output directory: D:/College/MENEA/FIXED_DATA/OUTPUT/20200228/shapematch

Matched files:
Pred 1 -> Obs obs_new_20200228_152411.png (Δ 0s)
Pred 2 -> Obs obs_new_20200228_152612.png (Δ 0s)
Pred 3 -> Obs obs_new_20200228_152812.png (Δ 1s)
Pred 4 -> Obs obs_new_20200228_153010.png (Δ 4s)
Pred 5 -> Obs obs_new_20200228_153211.png (Δ 4s)
Pred 6 -> Obs obs_new_20200228_153411.png (Δ 5s)
Pred 7 -> Obs obs_new_20200228_153612.png (Δ 5s)
Pred 8 -> Obs obs_new_20200228_153811.png (Δ 7s)
Pred 9 -> Obs obs_new_20200228_154012.png (Δ 7s)
Pred 10 -> Obs obs_new_20200228_154210.png (Δ 10s)
Pred 11 -> Obs obs_new_20200228_154411.png (Δ 10s)
Pred 12 -> Obs obs_new_20200228_154612.png (Δ 10s)
Pred 13 -> Obs obs_new_20200228_154812.png (Δ 11s)
Pred 14 -> Obs obs_new_20200228_155011.png (Δ 13s)
Pred 15 -> Obs obs_new_

In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import tensorflow as tf
from skimage.metrics import structural_similarity as ssim
from skimage.measure import label, regionprops

# Configure matplotlib styles
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

def find_matching_files(pred_dir, obs_dir):
    """Find matching prediction and observation files with improved timestamp pairing"""
    file_pairs = []
    
    # Extract and sort prediction files by sequence number
    pred_files = []
    for f in os.listdir(pred_dir):
        match = re.match(r'^(\d+)\.(png|jpg|jpeg)$', f, re.IGNORECASE)
        if match:
            sequence_num = int(match.group(1))
            pred_files.append((sequence_num, f))
    pred_files.sort(key=lambda x: x[0])
    
    if not pred_files:
        print(f"Error: No prediction files found in {pred_dir}")
        return file_pairs
    
    # Extract and sort observation files by timestamp
    obs_files = []
    for f in os.listdir(obs_dir):
        match = re.search(r'_(\d{6})\.(png|jpg|jpeg)$', f, re.IGNORECASE)
        if match:
            time_str = match.group(1)
            try:
                timestamp = (int(time_str[:2]) * 3600 + 
                           int(time_str[2:4]) * 60 + 
                           int(time_str[4:6]))
                obs_files.append((timestamp, f))
            except ValueError:
                continue
    obs_files.sort(key=lambda x: x[0])
    
    if not obs_files:
        print(f"Error: No observation files found in {obs_dir}")
        return file_pairs
    
    # Calculate dynamic time step (seconds between observations)
    time_step = 120  # default 2 minutes
    if len(obs_files) > 1:
        time_diffs = [obs_files[i+1][0]-obs_files[i][0] for i in range(len(obs_files)-1)]
        time_step = int(np.median(time_diffs))
    
    first_obs_time = obs_files[0][0]
    
    # Pair files with improved temporal matching
    for seq_num, pred_file in pred_files:
        pred_time = first_obs_time + (seq_num - 1) * time_step
        closest_obs = min(obs_files, key=lambda x: abs(x[0] - pred_time))
        time_diff = abs(closest_obs[0] - pred_time)
        
        # Only allow matches within 1.5x the time step
        if time_diff <= 1.5 * time_step:
            file_pairs.append({
                'timestep': seq_num,
                'pred_path': os.path.join(pred_dir, pred_file),
                'obs_path': os.path.join(obs_dir, closest_obs[1]),
                'time_diff': time_diff,
                'pred_time': pred_time,
                'obs_time': closest_obs[0],
                'obs_filename': closest_obs[1]
            })
    
    # Debug output
    print(f"\nMatched {len(file_pairs)}/{len(pred_files)} prediction files")
    print("Sample matches:")
    for pair in file_pairs[:5]:
        print(f"Pred {pair['timestep']} -> Obs {pair['obs_filename']} (Δ {pair['time_diff']}s)")
    
    return file_pairs

def load_and_preprocess(image_path, target_size=None):
    """Improved image loading with validation and preprocessing"""
    try:
        img = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
        if img is None:
            raise ValueError(f"Failed to load image: {image_path}")
        
        # Handle various image formats
        if len(img.shape) == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        elif img.shape[2] == 4:
            img = cv2.cvtColor(img, cv2.COLOR_BGRA2RGB)
        elif img.shape[2] == 1:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Resize if needed
        if target_size:
            img = cv2.resize(img, target_size, interpolation=cv2.INTER_AREA)
        
        return img
    except Exception as e:
        print(f"Error loading {image_path}: {str(e)}")
        return None

def calculate_ssim(img1, img2):
    """Calculate SSIM with improved validation"""
    if img1 is None or img2 is None:
        return 0.0
    
    try:
        # Convert to grayscale if needed
        if len(img1.shape) == 3:
            img1 = cv2.cvtColor(img1, cv2.COLOR_RGB2GRAY)
        if len(img2.shape) == 3:
            img2 = cv2.cvtColor(img2, cv2.COLOR_RGB2GRAY)
        
        # Ensure identical dimensions
        min_height = min(img1.shape[0], img2.shape[0])
        min_width = min(img1.shape[1], img2.shape[1])
        img1 = img1[:min_height, :min_width]
        img2 = img2[:min_height, :min_width]
        
        # Calculate dynamic data range
        data_range = max(img2.max() - img2.min(), img1.max() - img1.min())
        if data_range == 0:
            return 0.0
        
        # Use optimized window size
        win_size = min(7, min(img1.shape[0], img1.shape[1]) - 1)
        if win_size % 2 == 0:  # ensure odd window size
            win_size -= 1
        
        return ssim(img1, img2, 
                   data_range=data_range,
                   win_size=win_size,
                   channel_axis=None)
    except Exception as e:
        print(f"SSIM calculation failed: {str(e)}")
        return 0.0

def calculate_ms_ssim(img1, img2):
    """Improved MS-SSIM calculation with better normalization"""
    if img1 is None or img2 is None:
        return 0.0
    
    try:
        # Ensure identical dimensions
        min_height = min(img1.shape[0], img2.shape[0])
        min_width = min(img1.shape[1], img2.shape[1])
        img1 = img1[:min_height, :min_width]
        img2 = img2[:min_height, :min_width]
        
        # Convert to 3-channel if needed
        if len(img1.shape) == 2:
            img1 = np.stack([img1]*3, axis=-1)
        elif img1.shape[2] == 1:
            img1 = np.repeat(img1, 3, axis=-1)
            
        if len(img2.shape) == 2:
            img2 = np.stack([img2]*3, axis=-1)
        elif img2.shape[2] == 1:
            img2 = np.repeat(img2, 3, axis=-1)
        
        # Normalize to [0,1] range
        img1 = img1.astype(np.float32)
        img2 = img2.astype(np.float32)
        
        if img1.max() > 1.0 or img2.max() > 1.0:
            img1 /= 255.0
            img2 /= 255.0
        
        # Calculate MS-SSIM with optimized parameters
        ms_ssim = tf.image.ssim_multiscale(
            tf.convert_to_tensor(img1),
            tf.convert_to_tensor(img2),
            max_val=1.0,
            filter_size=11,
            filter_sigma=1.5,
            k1=0.01,
            k2=0.03
        ).numpy()
        
        return float(np.clip(ms_ssim, 0.0, 1.0))
        
    except Exception as e:
        print(f"MS-SSIM calculation failed: {str(e)}")
        return 0.0

def create_comparison_figure(pred_img, obs_img, metrics, timestep):
    """Enhanced comparison visualization with metrics"""
    fig = plt.figure(figsize=(16, 8))
    
    # Prediction subplot
    ax1 = plt.subplot(1, 2, 1)
    ax1.imshow(pred_img)
    ax1.set_title(f"Prediction (Timestep {timestep})", fontsize=12, pad=10)
    ax1.axis('off')
    
    # Observation subplot
    ax2 = plt.subplot(1, 2, 2)
    ax2.imshow(obs_img)
    ax2.set_title(f"Observation (Δ {metrics['time_diff']}s)", fontsize=12, pad=10)
    ax2.axis('off')
    
    # Metrics annotation
    metric_text = (f"SSIM: {metrics['ssim']:.3f}\n"
                  f"MS-SSIM: {metrics['ms_ssim']:.3f}\n"
                  f"Time Diff: {metrics['time_diff']}s")
    
    fig.text(0.5, 0.92, metric_text, 
             ha='center', va='center', fontsize=12,
             bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray'))
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    return fig

def create_timeseries_plot(timesteps, metrics):
    """Enhanced time series visualization"""
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Plot both SSIM and MS-SSIM
    ax.plot(timesteps, [m['ssim'] for m in metrics], 
            'o-', color='#2c7bb6', label='SSIM')
    ax.plot(timesteps, [m['ms_ssim'] for m in metrics], 
            's-', color='#d7191c', label='MS-SSIM')
    
    # Reference zones
    ref_zones = [
        (0.8, 1.0, '#d9f0a3', 'Excellent'),
        (0.5, 0.8, '#fee08b', 'Good'), 
        (0.0, 0.5, '#fc8d59', 'Poor')
    ]
    
    for ymin, ymax, color, label in ref_zones:
        ax.axhspan(ymin, ymax, facecolor=color, alpha=0.2)
    
    # Customize plot
    ax.set(xlabel='Timestep', ylabel='Similarity Score',
           title='Image Similarity Metrics Over Time',
           ylim=(0, 1.05))
    ax.legend(loc='upper right')
    ax.grid(True, linestyle='--', alpha=0.4)
    
    # Add value annotations
    for i, m in enumerate(metrics):
        ax.annotate(f"{m['ms_ssim']:.2f}", 
                   (timesteps[i], m['ms_ssim']),
                   textcoords="offset points",
                   xytext=(0, 5), ha='center')
    
    plt.tight_layout()
    return fig

def process_time_series(pred_dir, obs_dir, output_dir):
    """Main processing pipeline with improved error handling"""
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'comparisons'), exist_ok=True)
    
    # Find matching files
    file_pairs = find_matching_files(pred_dir, obs_dir)
    if not file_pairs:
        print("\nError: No valid file pairs found!")
        print("Prediction files:", os.listdir(pred_dir)[:5], "...")
        print("Observation files:", os.listdir(obs_dir)[:5], "...")
        return [], []
    
    results = []
    
    for pair in file_pairs:
        try:
            # Load and validate images
            pred_img = load_and_preprocess(pair['pred_path'])
            obs_img = load_and_preprocess(pair['obs_path'])
            
            if pred_img is None or obs_img is None:
                print(f"Skipping timestep {pair['timestep']} due to loading error")
                continue
            
            # Calculate metrics
            metrics = {
                'ssim': calculate_ssim(pred_img, obs_img),
                'ms_ssim': calculate_ms_ssim(pred_img, obs_img),
                'time_diff': pair['time_diff']
            }
            
            print(f"Timestep {pair['timestep']}: "
                  f"SSIM={metrics['ssim']:.3f}, "
                  f"MS-SSIM={metrics['ms_ssim']:.3f}")
            
            # Save comparison figure
            fig = create_comparison_figure(pred_img, obs_img, metrics, pair['timestep'])
            fig.savefig(
                os.path.join(output_dir, 'comparisons', f'comparison_{pair["timestep"]}.png'),
                bbox_inches='tight', dpi=150
            )
            plt.close(fig)
            
            results.append({
                'timestep': pair['timestep'],
                **metrics,
                'obs_time': pair['obs_time']
            })
            
        except Exception as e:
            print(f"Error processing timestep {pair['timestep']}: {str(e)}")
            continue
    
    # Save results if we have any
    if results:
        # Save time series plot
        ts_fig = create_timeseries_plot(
            [r['timestep'] for r in results],
            results
        )
        ts_fig.savefig(
            os.path.join(output_dir, 'similarity_metrics.png'),
            bbox_inches='tight', dpi=150
        )
        plt.close(ts_fig)
        
        # Export CSV
        csv_path = os.path.join(output_dir, 'results.csv')
        with open(csv_path, 'w') as f:
            f.write("Timestep,SSIM,MS-SSIM,ObservationTime,TimeDifference\n")
            for r in results:
                f.write(f"{r['timestep']},{r['ssim']:.4f},{r['ms_ssim']:.4f},"
                       f"{r['obs_time']},{r['time_diff']}\n")
        
        print(f"\nSuccessfully processed {len(results)} timesteps")
        print(f"Results saved to: {output_dir}")
        
        return [r['timestep'] for r in results], [r['ms_ssim'] for r in results]
    else:
        print("\nNo valid results generated")
        return [], []

if __name__ == "__main__":
    # Configure paths - update these for your system
    BASE_DIR = "D:/College/MENEA/FIXED_DATA/OUTPUT/20221109/"
    PRED_DIR = os.path.join(BASE_DIR, 'nowcasts')
    OBS_DIR = os.path.join(BASE_DIR, 'observations')
    OUTPUT_DIR = os.path.join(BASE_DIR, 'shapematch')
    
    print("\n" + "="*60)
    print("Advanced Nowcast Verification System".center(60))
    print("="*60)
    
    # Verify directories
    print("\nDirectory Configuration:")
    print(f"Predictions: {PRED_DIR}")
    print(f"Observations: {OBS_DIR}")
    print(f"Output: {OUTPUT_DIR}")
    
    try:
        # Run the processing pipeline
        timesteps, scores = process_time_series(
            pred_dir=PRED_DIR,
            obs_dir=OBS_DIR,
            output_dir=OUTPUT_DIR
        )
        
        if timesteps:
            plt.show()  # Display plots if running interactively
    except Exception as e:
        print(f"\nFatal error: {str(e)}")


            Advanced Nowcast Verification System            

Directory Configuration:
Predictions: D:/College/MENEA/FIXED_DATA/OUTPUT/20221109/nowcasts
Observations: D:/College/MENEA/FIXED_DATA/OUTPUT/20221109/observations
Output: D:/College/MENEA/FIXED_DATA/OUTPUT/20221109/shapematch

Matched 30/30 prediction files
Sample matches:
Pred 1 -> Obs obs_new_20221109_144813.png (Δ 0s)
Pred 2 -> Obs obs_new_20221109_145012.png (Δ 2s)
Pred 3 -> Obs obs_new_20221109_145213.png (Δ 2s)
Pred 4 -> Obs obs_new_20221109_145412.png (Δ 4s)
Pred 5 -> Obs obs_new_20221109_145613.png (Δ 4s)
Timestep 1: SSIM=0.762, MS-SSIM=0.491
Timestep 2: SSIM=0.756, MS-SSIM=0.486
Timestep 3: SSIM=0.755, MS-SSIM=0.485
Timestep 4: SSIM=0.759, MS-SSIM=0.491
Timestep 5: SSIM=0.762, MS-SSIM=0.497
Timestep 6: SSIM=0.761, MS-SSIM=0.495
Timestep 7: SSIM=0.763, MS-SSIM=0.494
Timestep 8: SSIM=0.761, MS-SSIM=0.492
Timestep 9: SSIM=0.761, MS-SSIM=0.493
Timestep 10: SSIM=0.764, MS-SSIM=0.492
Timestep 11: SSIM=0.761, MS-SSIM=0.491